# Interactive all-pass ring resonator

Interactive notebook for the **Photonic Devices 2026** course.

The model uses the all-pass transfer function

\[
H(\lambda)=\frac{t-a e^{-j\phi(\lambda)}}{1-ta e^{-j\phi(\lambda)}},
\qquad
\phi(\lambda)=\beta(\lambda)L,
\qquad
\beta(\lambda)=\frac{2\pi n_{\rm eff}(\lambda)}{\lambda}.
\]

The effective index is linearized around \(\lambda_0\). Using

\[
n_g=n_{\rm eff}-\lambda\frac{d n_{\rm eff}}{d\lambda},
\]

we write

\[
n_{\rm eff}(\lambda)\simeq n_{\rm eff,0}
+
\frac{n_{\rm eff,0}-n_g}{\lambda_0}(\lambda-\lambda_0).
\]

The sliders control \(t\), \(L\), propagation loss, \(n_{\rm eff,0}\), and \(n_g\).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Enable widgets in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

c0 = 299792458.0
lambda0 = 1550e-9  # reference wavelength [m]


In [ ]:
def ring_response(t=0.98, L_um=62.8, loss_dB_cm=28.0, neff0=2.45, ng=4.0):
    # ----- units -----
    L = L_um * 1e-6
    L_cm = L * 100.0
    R = L / (2*np.pi)

    # ----- linearized effective index -----
    dneff_dlambda = (neff0 - ng) / lambda0

    # ----- round-trip field attenuation -----
    # loss_dB_cm is a POWER loss coefficient in dB/cm
    a = 10**(-loss_dB_cm * L_cm / 20.0)

    # ----- analytical estimates -----
    FSR_f = c0 / (ng * L)
    FSR_lambda_0 = lambda0**2 / (ng * L)

    rt = t * a
    if rt <= 0:
        return

    FWHM_lambda = FSR_lambda_0 * (1 - rt) / (np.pi * np.sqrt(rt))
    Q = lambda0 / FWHM_lambda
    finesse = FSR_lambda_0 / FWHM_lambda

    # ----- spectral window: several FSRs around lambda0 -----
    span = 3.2 * FSR_lambda_0
    lam = np.linspace(lambda0 - span, lambda0 + span, 12000)

    neff = neff0 + dneff_dlambda * (lam - lambda0)
    beta = 2*np.pi*neff / lam
    phi = beta * L

    p = a * np.exp(-1j * phi)
    H = (t - p) / (1 - t*p)

    T = np.abs(H)**2
    phase = np.unwrap(np.angle(H))

    # ----- coupling regime -----
    tol = 2e-4
    if abs(t - a) < tol:
        regime = "critical coupling"
    elif t > a:
        regime = "undercoupled"
    else:
        regime = "overcoupled"

    Hres = (t - a) / (1 - t*a)
    Tres = np.abs(Hres)**2

    # ----- text output -----
    print(f"Round-trip length L        = {L_um:.2f} um")
    print(f"Equivalent radius R        = {R*1e6:.2f} um")
    print(f"n_eff(lambda0)             = {neff0:.4f}")
    print(f"n_g                        = {ng:.4f}")
    print(f"Propagation loss           = {loss_dB_cm:.2f} dB/cm")
    print(f"Round-trip amplitude a     = {a:.6f}")
    print(f"Self-coupling t            = {t:.6f}")
    print(f"Coupling regime            = {regime}")
    print()
    print(f"FSR                        = {FSR_f/1e9:.2f} GHz")
    print(f"FSR                        = {FSR_lambda_0*1e9:.3f} nm")
    print(f"FWHM                       = {FWHM_lambda*1e9:.4f} nm")
    print(f"Finesse                    = {finesse:.2f}")
    print(f"Q                          = {Q:.3e}")
    print(f"Transmission at resonance  = {Tres:.4f}")

    # ----- power transmission -----
    plt.figure(figsize=(9, 4))
    plt.plot(lam*1e9, T, linewidth=1.6)
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Power transmission")
    plt.title(f"All-pass ring — {regime}")
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.show()

    # ----- phase response -----
    plt.figure(figsize=(9, 4))
    plt.plot(lam*1e9, phase, linewidth=1.6)
    plt.xlabel("Wavelength [nm]")
    plt.ylabel("Phase of H [rad]")
    plt.title("Phase response")
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
style = {"description_width": "180px"}
layout = widgets.Layout(width="520px")

t_slider = widgets.FloatSlider(
    value=0.98, min=0.90, max=0.9995, step=0.0005,
    description="Self-coupling t",
    readout_format=".4f",
    continuous_update=False,
    style=style, layout=layout
)

L_slider = widgets.FloatSlider(
    value=62.8, min=20.0, max=300.0, step=0.2,
    description="Round-trip L [um]",
    readout_format=".1f",
    continuous_update=False,
    style=style, layout=layout
)

loss_slider = widgets.FloatSlider(
    value=28.0, min=0.1, max=100.0, step=0.1,
    description="Loss [dB/cm]",
    readout_format=".1f",
    continuous_update=False,
    style=style, layout=layout
)

neff_slider = widgets.FloatSlider(
    value=2.45, min=1.5, max=3.5, step=0.01,
    description="n_eff(lambda0)",
    readout_format=".2f",
    continuous_update=False,
    style=style, layout=layout
)

ng_slider = widgets.FloatSlider(
    value=4.00, min=2.0, max=5.5, step=0.01,
    description="n_g",
    readout_format=".2f",
    continuous_update=False,
    style=style, layout=layout
)

controls = widgets.VBox([
    t_slider,
    L_slider,
    loss_slider,
    neff_slider,
    ng_slider
])

out = widgets.interactive_output(
    ring_response,
    {
        "t": t_slider,
        "L_um": L_slider,
        "loss_dB_cm": loss_slider,
        "neff0": neff_slider,
        "ng": ng_slider,
    }
)

display(controls, out)


## What to explore

- Increase \(L\): the resonances become more closely spaced because the FSR decreases.
- Change \(n_g\): the resonance spacing changes directly.
- Change \(n_{\rm eff}\): the absolute resonance positions shift.
- Change the propagation loss: the round-trip amplitude \(a\) changes.
- Sweep \(t\) across \(a\): the ring moves from undercoupled to critical coupling and then to overcoupled.

At critical coupling, \(t=a\), the through-port transmission ideally vanishes at resonance.
